# Preprocessing Pipeline — Minh hoạ trực quan

Notebook này minh hoạ từng bước tiền xử lý dữ liệu từ bài báo JSON → cặp (ảnh, caption) → đặc trưng HDF5.

> **Lưu ý:** Các hình ảnh được sinh trước bởi script `gen_viz.py` và lưu vào thư mục `notebooks/pipeline/`. Notebook này hiển thị và giải thích các hình đó.


In [32]:
import os, sys, json
from pathlib import Path
from IPython.display import Image, display

# Resolve project root by walking upward until src/ found
PROJECT_ROOT = Path().resolve()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

# PNG files live in notebooks/pipeline/ — derive reliably from project root
NB_DIR = PROJECT_ROOT / "notebooks" / "pipeline"

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))

try:
    from dotenv import load_dotenv

    load_dotenv(PROJECT_ROOT / ".env", override=False)
except ImportError:
    pass

DATA_ROOT = (
    Path(os.environ["DATA_ROOT"]) if os.environ.get("DATA_ROOT") else PROJECT_ROOT
)
JSON_DIR = DATA_ROOT / "data" / "json"
HDF5_DIR = DATA_ROOT / "processed_data" / "hdf5"
PAIRS_DIR = HDF5_DIR / "pairs"

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"NB_DIR       : {NB_DIR}")
print(f"DATA_ROOT    : {DATA_ROOT}")
print(f"HDF5 exists  : {HDF5_DIR.exists()}")
print(f"Pairs exists : {PAIRS_DIR.exists()}")
print(f'PNG files    : {len(list(NB_DIR.glob("fig_p*.png")))} found')

PROJECT_ROOT : /Users/haila/My File/projects/fake-new-detection
NB_DIR       : /Users/haila/My File/projects/fake-new-detection/notebooks/pipeline
DATA_ROOT    : /Users/haila/Library/CloudStorage/GoogleDrive-latruonghai@gmail.com/My Drive/Thesis_Final/fake-news-data-for-thesis
HDF5 exists  : True
Pairs exists : True
PNG files    : 10 found


---

## Pipeline Tổng Quan

Sơ đồ toàn bộ pipeline từ JSON đến HDF5 features.


![Pipeline tổng quan](fig_p0_pipeline_overview.png)


---

## 1. Dữ liệu đầu vào: Bài báo JSON

Mỗi bài báo là một JSON object với các trường:

-   `id` — định danh duy nhất
-   `features` — `title`, `text`, `domain`, `image_count`, ...
-   `media.images[]` — danh sách ảnh (`path`, `caption`, `url`)
-   `metadata` — `source_url`, `crawl_timestamp`

Label được đính kèm theo từng bài từ dataset ViFactCheck.


![Cấu trúc JSON bài báo ViFactCheck](fig_p1_json_structure.png)

_Hình 1: Cấu trúc JSON bài báo ViFactCheck_


In [33]:
# Xem cấu trúc bài báo thực
articles = json.load(
    open(JSON_DIR / "news_data_vifactcheck_train_labeled.json", encoding="utf-8")
)
# Tìm bài có ảnh (dùng cấu trúc media.images)
sample = next(
    (a for a in articles if (a.get("media") or {}).get("images")), articles[0]
)
print(f"Tổng bài báo (train): {len(articles):,}")
print(f'\nID: {sample["id"]}')
feat = sample["features"]
print(f'domain : {feat.get("domain")}')
print(f'title  : {str(feat.get("title",""))[:80]}')
print(f'text   : {str(feat.get("text",""))[:100]}...')
print(f'image_count: {feat.get("image_count")}')
imgs = (sample.get("media") or {}).get("images") or []
for i, img in enumerate(imgs[:2]):
    print(f'  img[{i}] path   : {img.get("path","")}')
    print(f'  img[{i}] caption: {str(img.get("caption",""))[:90]}')

Tổng bài báo (train): 5,062

ID: b1233e8ace66c810
domain : baotintuc.vn
title  : 
text   : ...
image_count: 2
  img[0] path   : jpg/bao_tin_tuc/bao_tin_tuc_37194b2c7d.jpg
  img[0] caption: Cơ quan chức năng làm công tác kiểm tra tại Cơ sở nuôi trồng thủy sản Thăng Tiến tại huyện
  img[1] path   : jpg/bao_tin_tuc/bao_tin_tuc_e69ac75a69.jpg
  img[1] caption: Cơ quan chức năng làm công tác kiểm tra tại Cơ sở nuôi trồng thủy sản Thăng Tiến tại huyện


---

## 2. Trích xuất cặp (image, caption) hợp lệ

**PairExtractor** duyệt qua từng bài báo:

1. Ghép `media.images[].path` → đường dẫn file JPG
2. Làm sạch caption (xoá HTML entities, credit ảnh)
3. Lọc bỏ cặp không hợp lệ theo tiêu chí:

| Filter            | Điều kiện                                |
| ----------------- | ---------------------------------------- |
| `no_image`        | File JPG không tồn tại                   |
| `no_caption`      | Caption rỗng sau khi làm sạch            |
| `credit_only`     | Caption chỉ là tên nguồn (AFP, TTXVN...) |
| `too_short`       | Độ dài < `min_caption_len=5`             |
| ✅ **valid pair** | Qua tất cả các bộ lọc                    |


![Làm sạch caption trước/sau](fig_p2_caption_cleaning.png)

_Hình 2: Làm sạch caption trước/sau_


![Số cặp hợp lệ / bị lọc bỏ theo split](fig_p3_pair_extraction.png)

_Hình 3: Số cặp hợp lệ / bị lọc bỏ theo split_


In [34]:
from preprocessing.coolant.pair_extractor import clean_caption
import pandas as pd

examples = [
    "Cán bộ kiểm tra tại hiện trường tỉnh Đắk Lắk. Ảnh: TTXVN phát",
    "Học sinh tham gia kỳ thi THPT 2023 tại TP.HCM. Ảnh minh họa: VnExpress",
    "Hội thảo quốc tế về khoa học. Nguồn ảnh: Reuters",
    "AFP",
]
pd.set_option("display.max_colwidth", 80)
df = pd.DataFrame(
    [
        {
            "Trước": raw,
            "Sau": clean_caption(raw),
            "Đã thay đổi": raw != clean_caption(raw),
        }
        for raw in examples
    ]
)
display(df)

,Trước,Sau,Đã thay đổi
0,Cán bộ kiểm tra tại hiện trường tỉnh Đắk Lắk. Ảnh: TTXVN phát,Cán bộ kiểm tra tại hiện trường tỉnh Đắk Lắk,True
1,Học sinh tham gia kỳ thi THPT 2023 tại TP.HCM. Ảnh minh họa: VnExpress,Học sinh tham gia kỳ thi THPT 2023 tại TP.HCM,True
2,Hội thảo quốc tế về khoa học. Nguồn ảnh: Reuters,Hội thảo quốc tế về khoa học. Nguồn,True
3,AFP,AFP,False


In [35]:
# Đọc pairs cache
pair_data = {}
for split in ["train", "dev", "test"]:
    p = PAIRS_DIR / f"pairs_{split}.json"
    if p.exists():
        pair_data[split] = json.load(open(p, encoding="utf-8"))
        print(f"{split:5}: {len(pair_data[split]):,} cặp hợp lệ")

# Mẫu cặp
if "train" in pair_data:
    p0 = pair_data["train"][0]
    print(f"\nMẫu cặp train[0]:")
    for k, v in p0.items():
        print(f"  {k}: {str(v)[:90]}")

train: 6,724 cặp hợp lệ
dev  : 862 cặp hợp lệ
test : 2,053 cặp hợp lệ

Mẫu cặp train[0]:
  image_path: /Users/haila/Library/CloudStorage/GoogleDrive-latruonghai@gmail.com/My Drive/Thesis_Final/
  caption: Cơ quan chức năng làm công tác kiểm tra tại Cơ sở nuôi trồng thủy sản Thăng Tiến tại huyện
  article_idx: 2
  folder_path: bao_tin_tuc/bao_tin_tuc_37194b2c7d.jpg
  pair_text: Cơ quan chức năng làm công tác kiểm tra tại Cơ sở nuôi trồng thủy sản Thăng Tiến tại huyện
  title: 
  source_url: https://baotintuc.vn/an-ninh-trat-tu/bat-tam-giam-doi-tuong-chon-lap-hon-600-tan-chat-thai
  source_label: 


---

## 3. Tiền xử lý văn bản (Text Pipeline)

```
caption thô
   ↓  clean_text()     xoá HTML, credit, URL, dấu câu, số → lowercase
   ↓  segment_words()  'học sinh' → 'học_sinh'  (underthesea)
   ↓  tokenizer()      chuỗi → input_ids, attention_mask  (max_length=128)
   ↓  bert_model()     → last_hidden_state  shape (128, 768)
```


![Các bước làm sạch văn bản](fig_p4_text_cleaning.png)

_Hình 4: Các bước làm sạch văn bản_


![PhoBERT tokenization](fig_p5_tokenization.png)

_Hình 5: PhoBERT tokenization — không vs có word segmentation_


In [36]:
# Demo tokenizer
try:
    from transformers import AutoTokenizer

    tok = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")

    texts = {
        "Không seg": "học sinh việt nam học tiếng anh rất chăm chỉ",
        "Có seg   ": "học_sinh việt_nam học tiếng_anh rất chăm_chỉ",
    }
    for label, text in texts.items():
        enc = tok(
            text,
            return_tensors="pt",
            max_length=16,
            padding="max_length",
            truncation=True,
        )
        tokens = tok.convert_ids_to_tokens(enc["input_ids"][0].tolist())
        n_real = sum(1 for t in tokens if t not in ["<s>", "</s>", "<pad>"])
        print(f"{label}  ({n_real} tokens): {tokens}")
except Exception as e:
    print(f"Tokenizer: {e}")

Không seg  (10 tokens): ['<s>', 'học', 'sinh', 'việt', 'nam', 'học', 'tiếng', 'anh', 'rất', 'chăm', 'chỉ', '</s>', '<pad>', '<pad>', '<pad>', '<pad>']
Có seg     (8 tokens): ['<s>', 'học_sinh', 'việt_@@', 'nam', 'học', 'tiếng_@@', 'anh', 'rất', 'chăm_chỉ', '</s>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>']


---

## 4. Tiền xử lý ảnh (Image Pipeline)

```
file.jpg
   ↓  PIL.open().convert('RGB')
   ↓  Resize(256,256)
   ↓  CenterCrop(224,224)
   ↓  ToTensor()          → float32 [0, 1]
   ↓  Normalize(ImageNet) → zero-centred
   ↓  ResNet50 (avgpool, no FC layer)
   →  feature vector  shape (2048,)
```


![Image preprocessing pipeline](fig_p6_image_pipeline.png)

_Hình 6: Image preprocessing — ảnh thực từ dataset_


---

## 5. Đặc trưng HDF5 — Cấu trúc và Nội dung

| Dataset            | Shape           | Nội dung                  |
| ------------------ | --------------- | ------------------------- |
| `caption_features` | `(N, 128, 768)` | PhoBERT last_hidden_state |
| `image_features`   | `(N, 2048)`     | ResNet50 avgpool          |
| `article_ids`      | `(N,)`          | Article index             |
| `source_labels`    | `(N,)`          | Label string              |
| `source_urls`      | `(N,)`          | URL nguồn                 |
| `image_paths`      | `(N,)`          | Đường dẫn ảnh             |


![Cấu trúc HDF5 trên 3 splits](fig_p7_hdf5_structure.png)

_Hình 7: Cấu trúc HDF5 trên 3 splits_


![Phân bố giá trị đặc trưng](fig_p8_feature_distribution.png)

_Hình 8: Phân bố giá trị đặc trưng (200 mẫu)_


In [37]:
import h5py

for split in ["train", "dev", "test"]:
    h5p = HDF5_DIR / f"coolant_{split}.h5"
    if not h5p.exists():
        print(f"{split}: not found")
        continue
    with h5py.File(h5p, "r") as f:
        n = int(f.attrs["n_samples"])
        cs = tuple(f["caption_features"].shape)
        is_ = tuple(f["image_features"].shape)
        mb = h5p.stat().st_size / 1024**2
    print(f"{split:5}: {n:,} pairs  cap={cs}  img={is_}  {mb:.0f} MB")

train: 6,724 pairs  cap=(6724, 128, 768)  img=(6724, 2048)  2577 MB
dev  : 862 pairs  cap=(862, 128, 768)  img=(862, 2048)  330 MB
test : 2,053 pairs  cap=(2053, 128, 768)  img=(2053, 2048)  787 MB


---

## 6. Tổng kết Pipeline

Funnel chart: số lượng mẫu qua từng bước xử lý.


![Pipeline funnel](fig_p9_pipeline_funnel.png)

_Hình 9: Pipeline funnel — JSON → cặp hợp lệ → đặc trưng HDF5_


In [ ]:
import pandas as pd

summary = [
    {
        "Split": "train",
        "Bài báo": 5062,
        "Ảnh tổng": 6948,
        "Cặp hợp lệ": 6724,
        "Loại bỏ": 224,
        "Tỉ lệ": "96.8%",
        "HDF5": "2,576 MB",
    },
    {
        "Split": "dev",
        "Bài báo": 723,
        "Ảnh tổng": 896,
        "Cặp hợp lệ": 862,
        "Loại bỏ": 34,
        "Tỉ lệ": "96.2%",
        "HDF5": "330 MB",
    },
    {
        "Split": "test",
        "Bài báo": 1447,
        "Ảnh tổng": 2100,
        "Cặp hợp lệ": 2053,
        "Loại bỏ": 47,
        "Tỉ lệ": "97.8%",
        "HDF5": "787 MB",
    },
    {
        "Split": "TOTAL",
        "Bài báo": 7232,
        "Ảnh tổng": 9944,
        "Cặp hợp lệ": 9639,
        "Loại bỏ": 305,
        "Tỉ lệ": "96.9%",
        "HDF5": "3,693 MB",
    },
]
df = pd.DataFrame(summary)
display(
    df.style.hide(axis="index")
    .set_caption("Bảng tổng kết preprocessing pipeline")
    .applymap(
        lambda v: (
            "font-weight:bold;background:#d4edda" if str(v).startswith("TOT") else ""
        ),
        subset=["Split"],
    )
)

/var/folders/rh/5t55t1152fjbnbkbpyk4_t6c0000gn/T/ipykernel_89548/4274376878.py:45: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  .applymap(


Split,Bài báo,Ảnh tổng,Cặp hợp lệ,Loại bỏ,Tỉ lệ,HDF5
train,5062,6948,6724,224,96.8%,"2,576 MB"
dev,723,896,862,34,96.2%,330 MB
test,1447,2100,2053,47,97.8%,787 MB
TOTAL,7232,9944,9639,305,96.9%,"3,693 MB"


: 

---

## Tóm tắt các bước

| Bước | Module                           | Input → Output                               |
| ---- | -------------------------------- | -------------------------------------------- |
| 1    | JSON loader                      | `*.json` → list of articles                  |
| 2    | `PairExtractor`                  | articles → `(image_path, caption)[]`         |
| 3    | `clean_caption`                  | caption raw → caption sạch                   |
| 4    | `TextPreprocessor.clean_text`    | text → lowercase, no HTML/punct/number       |
| 5    | `TextPreprocessor.segment_words` | text → word-segmented (`học_sinh`)           |
| 6    | `AutoTokenizer` PhoBERT          | text → `input_ids` (128,)                    |
| 7    | `AutoModel` PhoBERT              | `input_ids` → `last_hidden_state` (128, 768) |
| 8    | `ImagePreprocessor` ResNet50     | JPG → feature vector (2048,)                 |
| 9    | `h5py.File`                      | features → `coolant_{split}.h5`              |

**Notebook tiếp theo:** `03_coolant_training.ipynb`
